### **Bank Management System**

In [ ]:
import os
from datetime import datetime


class Account:

    def __init__(self, account_no, name, pin, balance):
        self.account_no = account_no
        self.name = name
        self.pin = pin
        self.balance = float(balance)

    def to_record(self):
        return f"{self.account_no},{self.name},{self.pin},{self.balance}\n"


class Transaction:

    FILE_NAME = "transactions.txt"

    @staticmethod
    def add(account_no, transaction_type, amount):

        with open(Transaction.FILE_NAME, "a") as file:

            date_time = datetime.now().strftime("%d-%m-%Y %H:%M:%S")

            file.write(
                f"{date_time},{account_no},{transaction_type},{amount}\n"
            )


class Bank:

    ACCOUNT_FILE = "accounts.txt"

    def __init__(self):

        if not os.path.exists(self.ACCOUNT_FILE):
            open(self.ACCOUNT_FILE, "w").close()

        if not os.path.exists(Transaction.FILE_NAME):
            open(Transaction.FILE_NAME, "w").close()

    # ----------------------------
    # Load Accounts
    # ----------------------------

    def load_accounts(self):

        accounts = []

        with open(self.ACCOUNT_FILE, "r") as file:

            for line in file:

                if line.strip():

                    acc_no, name, pin, balance = line.strip().split(",")

                    accounts.append(
                        Account(acc_no, name, pin, balance)
                    )

        return accounts

    # ----------------------------
    # Save Accounts
    # ----------------------------

    def save_accounts(self, accounts):

        with open(self.ACCOUNT_FILE, "w") as file:

            for account in accounts:

                file.write(account.to_record())

    # ----------------------------
    # Validate Account Number
    # ----------------------------

    def validate_account_number(self, account_no):

        if len(account_no) != 4:

            return False

        if not account_no.isdigit():

            return False

        return True

    # ----------------------------
    # Validate PIN
    # ----------------------------

    def validate_pin(self, pin):

        if len(pin) != 4:

            return False

        if not pin.isdigit():

            return False

        return True

    # ----------------------------
    # Find Account
    # ----------------------------

    def find_account(self, account_no):

        accounts = self.load_accounts()

        for account in accounts:

            if account.account_no == account_no:

                return account

        return None

    # ----------------------------
    # Verify PIN
    # ----------------------------

    def verify_pin(self, account_no, pin):

        account = self.find_account(account_no)

        if account:

            return account.pin == pin

        return False

    # ----------------------------
    # Check Duplicate Account
    # ----------------------------

    def account_exists(self, account_no):

        account = self.find_account(account_no)

        return account is not None

    # ----------------------------
    # Display Account
    # ----------------------------

    def display_account(self, account):

        print("-" * 40)
        print("Account Number :", account.account_no)
        print("Name           :", account.name)
        
    
    # ----------------------------    
    # Create Account
    # ----------------------------

    def create_account(self):

        account_no = input("Enter 4-digit Account Number: ")
    
        if not self.validate_account_number(account_no):
            print("Invalid Account Number. It must contain exactly 4 digits.")
            return
    
        if self.account_exists(account_no):
            print("Account already exists.")
            return
    
        # -------- Name Validation --------
        while True:
    
            name = input("Enter Full Name (First Middle Last): ").strip()
    
            words = name.split()
    
            if len(words) != 3:
                print("Name must contain exactly 3 words (First Middle Last).")
                continue
    
            valid = True
    
            for word in words:
    
                if not word.isalpha():
                    valid = False
                    break
    
            if not valid:
                print("Name should contain only alphabets.")
                continue
    
            break
    
        # -------- PIN --------
        pin = input("Enter 4-digit PIN: ")
    
        if not self.validate_pin(pin):
            print("PIN must contain exactly 4 digits.")
            return
    
        confirm_pin = input("Confirm PIN: ")
    
        if pin != confirm_pin:
            print("PIN Mismatch.")
            return
    
        try:
            balance = float(input("Enter Initial Balance: "))
    
            if balance < 0:
                print("Balance cannot be negative.")
                return
    
        except ValueError:
            print("Invalid Amount.")
            return
    
        account = Account(account_no, name, pin, balance)
    
        accounts = self.load_accounts()
        accounts.append(account)
        self.save_accounts(accounts)
    
        Transaction.add(account_no, "Account Created", balance)
    
        print("Account Created Successfully.")

    
    # ----------------------------
    # Search Account
    # ----------------------------

    def search_account(self):

        account_no = input("Enter Account Number: ")

        account = self.find_account(account_no)

        if account:
            self.display_account(account)
        else:
            print("Account does not exist.")

    # ----------------------------
    # Deposit
    # ----------------------------

    def deposit(self):

        account_no = input("Enter Account Number: ")

        if not self.account_exists(account_no):
            print("Account does not exist.")
            return

        try:
            amount = float(input("Enter Amount to Deposit: "))

            if amount <= 0:
                print("Amount must be greater than zero.")
                return

        except ValueError:
            print("Invalid Amount.")
            return

        accounts = self.load_accounts()

        for account in accounts:

            if account.account_no == account_no:

                account.balance += amount

                Transaction.add(account_no, "Deposit", amount)

                break

        self.save_accounts(accounts)

        print("Amount Deposited Successfully.")

    # ----------------------------
    # Withdraw
    # ----------------------------

    def withdraw(self):

        account_no = input("Enter Account Number: ")

        if not self.account_exists(account_no):
            print("Account does not exist.")
            return

        pin = input("Enter PIN: ")

        if not self.verify_pin(account_no, pin):
            print("Incorrect PIN.")
            return

        try:
            amount = float(input("Enter Amount to Withdraw: "))

            if amount <= 0:
                print("Amount must be greater than zero.")
                return

        except ValueError:
            print("Invalid Amount.")
            return

        accounts = self.load_accounts()

        for account in accounts:

            if account.account_no == account_no:

                if account.balance < amount:
                    print("Insufficient Balance.")
                    return

                account.balance -= amount

                Transaction.add(account_no, "Withdraw", amount)

                break

        self.save_accounts(accounts)

        print("Withdrawal Successful.")

    # ----------------------------
    # Check Balance
    # ----------------------------

    def check_balance(self):

        account_no = input("Enter Account Number: ")

        if not self.account_exists(account_no):
            print("Account does not exist.")
            return

        pin = input("Enter PIN: ")

        if not self.verify_pin(account_no, pin):
            print("Incorrect PIN.")
            return

        account = self.find_account(account_no)

        print(f"\nAvailable Balance : ₹{account.balance:.2f}")
        print("Balance        :", account.balance)
        print("-" * 40)


    # ----------------------------
    # Transfer Money
    # ----------------------------

    def transfer_money(self):

        sender = input("Enter Sender Account Number: ")
        receiver = input("Enter Receiver Account Number: ")

        if sender == receiver:
            print("Sender and Receiver account cannot be the same.")
            return

        if not self.account_exists(sender):
            print("Sender Account does not exist.")
            return

        if not self.account_exists(receiver):
            print("Receiver Account does not exist.")
            return

        pin = input("Enter Sender PIN: ")

        if not self.verify_pin(sender, pin):
            print("Incorrect PIN.")
            return

        try:
            amount = float(input("Enter Amount to Transfer: "))

            if amount <= 0:
                print("Amount should be greater than zero.")
                return

        except ValueError:
            print("Invalid Amount.")
            return

        accounts = self.load_accounts()

        sender_account = None
        receiver_account = None

        for account in accounts:

            if account.account_no == sender:
                sender_account = account

            elif account.account_no == receiver:
                receiver_account = account

        if sender_account.balance < amount:
            print("Insufficient Balance.")
            return

        sender_account.balance -= amount
        receiver_account.balance += amount

        self.save_accounts(accounts)

        Transaction.add(sender, "Transfer Sent", amount)
        Transaction.add(receiver, "Transfer Received", amount)

        print("Money Transferred Successfully.")

    # ----------------------------
    # Transaction History
    # ----------------------------

    def transaction_history(self):

        account_no = input("Enter Account Number: ")

        if not self.account_exists(account_no):
            print("Account does not exist.")
            return

        print("\nTransaction History")
        print("-" * 70)

        found = False

        with open(Transaction.FILE_NAME, "r") as file:

            for line in file:

                date, acc, trans_type, amount = line.strip().split(",")

                if acc == account_no:

                    print(f"{date} | {trans_type} | ₹{amount}")

                    found = True

        if not found:
            print("No Transactions Found.")

    # ----------------------------
    # Change PIN
    # ----------------------------

    def change_pin(self):

        account_no = input("Enter Account Number: ")

        if not self.account_exists(account_no):
            print("Account does not exist.")
            return

        old_pin = input("Enter Old PIN: ")

        if not self.verify_pin(account_no, old_pin):
            print("Incorrect PIN.")
            return

        new_pin = input("Enter New 4-digit PIN: ")

        if not self.validate_pin(new_pin):
            print("PIN must be exactly 4 digits.")
            return

        confirm_pin = input("Confirm New PIN: ")

        if new_pin != confirm_pin:
            print("PIN Mismatch.")
            return

        accounts = self.load_accounts()

        for account in accounts:

            if account.account_no == account_no:

                account.pin = new_pin
                break

        self.save_accounts(accounts)

        print("PIN Changed Successfully.")

    # ----------------------------
    # Delete Account
    # ----------------------------

    def delete_account(self):

        account_no = input("Enter Account Number: ")

        if not self.account_exists(account_no):
            print("Account does not exist.")
            return

        pin = input("Enter PIN: ")

        if not self.verify_pin(account_no, pin):
            print("Incorrect PIN.")
            return

        choice = input("Are you sure you want to delete this account? (Y/N): ")

        if choice.upper() != "Y":
            print("Deletion Cancelled.")
            return

        accounts = self.load_accounts()

        accounts = [
            account for account in accounts
            if account.account_no != account_no
        ]

        self.save_accounts(accounts)

        print("Account Deleted Successfully.")

    # ----------------------------
    # View All Accounts
    # ----------------------------

    def view_all_accounts(self):

        accounts = self.load_accounts()

        if not accounts:
            print("No Accounts Found.")
            return

        print("\nAll Accounts")
        print("-" * 60)

        for account in accounts:

            print(f"Account No : {account.account_no}")
            print(f"Name       : {account.name}")
            print(f"Balance    : ₹{account.balance:.2f}")
            print("-" * 60)

# ==========================
# Main Program
# ==========================

def main():

    bank = Bank()

    while True:

        print("\n" + "=" * 50)
        print("        BANK MANAGEMENT SYSTEM")
        print("=" * 50)

        print("1. Create Account")
        print("2. Search Account")
        print("3. Deposit")
        print("4. Withdraw")
        print("5. Check Balance")
        print("6. Transfer Money")
        print("7. Transaction History")
        print("8. Change PIN")
        print("9. Delete Account")
        print("10. View All Accounts")
        print("11. Exit")

        choice = input("\nEnter Your Choice : ")

        if choice == "1":
            bank.create_account()

        elif choice == "2":
            bank.search_account()

        elif choice == "3":
            bank.deposit()

        elif choice == "4":
            bank.withdraw()

        elif choice == "5":
            bank.check_balance()

        elif choice == "6":
            bank.transfer_money()

        elif choice == "7":
            bank.transaction_history()

        elif choice == "8":
            bank.change_pin()

        elif choice == "9":
            bank.delete_account()

        elif choice == "10":
            bank.view_all_accounts()

        elif choice == "11":
            print("\nThank You for using Bank Management System.")
            print("Good Bye!")
            break

        else:
            print("Invalid Choice. Please select between 1 and 11.")


# ==========================
# Program Starts Here
# ==========================

if __name__ == "__main__":
    main()


        BANK MANAGEMENT SYSTEM
1. Create Account
2. Search Account
3. Deposit
4. Withdraw
5. Check Balance
6. Transfer Money
7. Transaction History
8. Change PIN
9. Delete Account
10. View All Accounts
11. Exit



Enter Your Choice :  1
Enter 4-digit Account Number:  1234
Enter Full Name (First Middle Last):  Ashwini Ajit Chavan
Enter 4-digit PIN:  1234
Confirm PIN:  1234
Enter Initial Balance:  10000


Account Created Successfully.

        BANK MANAGEMENT SYSTEM
1. Create Account
2. Search Account
3. Deposit
4. Withdraw
5. Check Balance
6. Transfer Money
7. Transaction History
8. Change PIN
9. Delete Account
10. View All Accounts
11. Exit



Enter Your Choice :  1
Enter 4-digit Account Number:  7890
Enter Full Name (First Middle Last):  Sagar S Rokade
Enter 4-digit PIN:  7890
Confirm PIN:  7890
Enter Initial Balance:  200000


Account Created Successfully.

        BANK MANAGEMENT SYSTEM
1. Create Account
2. Search Account
3. Deposit
4. Withdraw
5. Check Balance
6. Transfer Money
7. Transaction History
8. Change PIN
9. Delete Account
10. View All Accounts
11. Exit


...

Money Transferred Successfully.

        BANK MANAGEMENT SYSTEM
1. Create Account
2. Search Account
3. Deposit
4. Withdraw
5. Check Balance
6. Transfer Money
7. Transaction History
8. Change PIN
9. Delete Account
10. View All Accounts
11. Exit



Enter Your Choice :  7
Enter Account Number:  123


Account does not exist.

        BANK MANAGEMENT SYSTEM
1. Create Account
2. Search Account
3. Deposit
4. Withdraw
5. Check Balance
6. Transfer Money
7. Transaction History
8. Change PIN
9. Delete Account
10. View All Accounts
11. Exit



Enter Your Choice :  7
Enter Account Number:  1234



Transaction History
----------------------------------------------------------------------
30-06-2026 14:50:42 | Account Created | ₹10000.0
30-06-2026 14:51:21 | Deposit | ₹1000.0
30-06-2026 14:51:43 | Transfer Received | ₹50000.0

        BANK MANAGEMENT SYSTEM
1. Create Account
2. Search Account
3. Deposit
4. Withdraw
5. Check Balance
6. Transfer Money
7. Transaction History
8. Change PIN
9. Delete Account
10. View All Accounts
11. Exit
